# Tier 2 — z-score on store×month 5%

After Tier 1 MVP (blacklist + freq store-day), flag store-months with top-box rate outside network mean ± 3σ.

In [ ]:
##Шаг A — загрузка

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]

from db.config import get_settings
get_settings.cache_clear()
from db.connection import get_engine
from fraud_guard.tier1 import (
    Tier1Config, apply_tier1, filter_answered_metric_rows,
    keep_clean_rows, top_box_rate,
)

engine = get_engine()
df = pd.read_sql("SELECT * FROM dbo.TargetsByMetrics_RateGetAnswers", engine)
print("shape:", df.shape)

In [ ]:
##Шаг B — Tier 1 MVP (база для Tier 2)

In [ ]:
work = filter_answered_metric_rows(df)
flagged = apply_tier1(work, config=Tier1Config(enable_always_topbox=False))
clean = keep_clean_rows(flagged)

print("answered:", len(work), "top-box:", round(top_box_rate(work), 4))
print("after Tier1 MVP:", len(clean), "top-box:", round(top_box_rate(clean), 4))

In [ ]:
##Шаг C — панель store×month + z-score

#z>3 сверху = 0 (порог 101% нереален)
#z>2 = 138, ≥90% = 135 — вот рабочие кандидаты
#z<−3 = 5 — слишком низкий score, не fraud

In [ ]:
def five_pct(s):
    return 100.0 * (s == 5).mean()

panel = (
    clean.groupby(["PrintStore", "Year", "Month"], dropna=False)["Answer_Value"]
    .agg(volume="count", five_pct=five_pct)
    .reset_index()
)
panel = panel[panel["volume"] >= 30].copy()

mu = panel["five_pct"].mean()
sigma = panel["five_pct"].std(ddof=0)
panel["z"] = (panel["five_pct"] - mu) / sigma

print("panel rows (vol>=30):", len(panel))
print("mean:", round(mu, 2), "std:", round(sigma, 2))
print("mean+3*std:", round(mu + 3 * sigma, 2), "← если >100, z>3 сверху почти невозможен")
print("z > 3:", int((panel["z"] > 3).sum()))
print("z > 2:", int((panel["z"] > 2).sum()))
print("five_pct >= 90:", int((panel["five_pct"] >= 90).sum()))
print("z < -3:", int((panel["z"] < -3).sum()))
print(panel["z"].describe(percentiles=[0.5, 0.95, 0.99]))

In [ ]:
## Шаг D — кто эти «высокие» store-months

In [ ]:
high = panel[(panel["z"] > 2) | (panel["five_pct"] >= 90)].sort_values("five_pct", ascending=False)
print("high candidates:", len(high))
print(high.head(15).to_string(index=False))
print("\nvolume in high:", int(high["volume"].sum()),
      "share of panel volume:", round(high["volume"].sum() / panel["volume"].sum(), 4))

In [ ]:
## Шаг E — эффект на сеть (используем ваш high)

In [ ]:
high_keys = set(zip(high["PrintStore"], high["Year"], high["Month"]))
mask = [
    (s, y, m) in high_keys
    for s, y, m in zip(clean["PrintStore"], clean["Year"], clean["Month"])
]

after_t2 = clean.loc[[not x for x in mask]].copy()

print("before (Tier1):", len(clean), "top-box:", round(top_box_rate(clean), 4))
print("dropped rows:", int(sum(mask)), "share:", round(sum(mask) / len(clean), 4))
print("after Tier2:", len(after_t2), "top-box:", round(top_box_rate(after_t2), 4))
print("top-box in dropped:", round(top_box_rate(clean.loc[mask]), 4))
print("network delta pp:", round(100 * (top_box_rate(after_t2) - top_box_rate(clean)), 2))

In [ ]:
## Шаг F — итоговая сводка pipeline

In [ ]:
print("=== pipeline ===")
print("0 answered Q10012: ", len(work), " top-box:", round(top_box_rate(work), 4))
print("1 Tier1 MVP:       ", len(clean), " top-box:", round(top_box_rate(clean), 4))
print("2 Tier2 high z/90: ", len(after_t2), " top-box:", round(top_box_rate(after_t2), 4))
print("total dropped:", len(work) - len(after_t2),
      "share:", round((len(work) - len(after_t2)) / len(work), 4))
print("total delta pp:", round(100 * (top_box_rate(after_t2) - top_box_rate(work)), 2))